In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn import tree

In [ ]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn import tree
import numpy as np

#데이터셋 불러오기
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 고윳값 Drop
X_train=train_df.drop(["index","name","health_status"],axis=1)
X_test=test_df.drop(["index","name"],axis=1)
y_train=train_df['health_status']

# 주요 피처 리스트
important_features = ['iron_lvl', 'stature_inch', 'fat_content', 'age_enzyme_ratio', 'Total_Cholesterol']

# 상호작용 피처 생성 함수
def create_interaction_features(df):
    
    # 추가 피처 생성 (주어진 코드대로)
    df['bmi'] = df['mass_kg'] / ((df['stature_inch'] + 100) / 100) ** 2
    df['age'] = 2024 - df['birth'] + 1
    df['bp'] = df['bp_high'] - df['bp_low']
    df['bp_tf'] = (df['bp_high'] > 130) & (df['bp_low'] > 85)
    df['bp_mean'] = (df['bp_high'] + df['bp_low']) / 2
    df['Waist_to_Height_Ratio'] = df['midsection_inch'] / df['stature_inch']
    df['LDL_to_HDL_Ratio'] = df['chol_bad'] / df['chol_good']
    df['Total_Cholesterol'] = df['chol_good'] + df['chol_bad']
    df['Enzyme_Avg'] = df[['enzyme_1', 'enzyme_2', 'enzyme_3']].mean(axis=1)
    df['Vision_Imbalance'] = (df['vision_l'] != df['vision_r'])
    df['vision_mean'] = (df['vision_l'] + df['vision_r']) / 2
    df['Metabolic_Syndrome'] = (
        (df['bp_high'] > 130) |
        (df['bp_low'] > 85) |
        (df['chol_bad'] > 100) |
        (df['midsection_inch'] > 35) |
        (df['sugar_lvl'] > 100)
    )
    df['waist_to_age_ratio'] = df['midsection_inch'] / df['age']
    df['audio_problem'] = ((df['audio_l'] != 'N') | (df['audio_r'] != 'N')).astype(int)
    df['urine_marker_flag'] = (df['urine_marker'] > 0).astype(int)
    df['age_enzyme_ratio'] = df['Enzyme_Avg'] / df['age']
    # 기본적인 상호작용 피처들
    df['iron_fat'] = df['iron_lvl'] * df['fat_content']
    df['stature_age'] = df['stature_inch'] * df['age_enzyme_ratio']
    df['cholesterol_iron'] = df['Total_Cholesterol'] * df['iron_lvl']

    # 나눗셈 상호작용
    df['iron_per_fat'] = df['iron_lvl'] / (df['fat_content'] + 1e-6)
    df['stature_per_cholesterol'] = df['stature_inch'] / (df['Total_Cholesterol'] + 1e-6)
    df['age_per_iron'] = df['age_enzyme_ratio'] / (df['iron_lvl'] + 1e-6)

    # 덧셈 상호작용
    df['iron_plus_fat'] = df['iron_lvl'] + df['fat_content']
    df['age_plus_cholesterol'] = df['age_enzyme_ratio'] + df['Total_Cholesterol']

    # 뺄셈 상호작용
    df['iron_minus_fat'] = df['iron_lvl'] - df['fat_content']
    df['cholesterol_minus_stature'] = df['Total_Cholesterol'] - df['stature_inch']

    # 비선형 상호작용 (제곱, 루트)
    df['iron_squared'] = df['iron_lvl'] ** 2
    df['sqrt_cholesterol'] = df['Total_Cholesterol'] ** 0.5
    
    # 곱셈 기반 상호작용
    df['iron_stature'] = df['iron_lvl'] * df['stature_inch']
    df['age_iron_squared'] = df['age_enzyme_ratio'] * df['iron_squared']
    df['cholesterol_iron'] = df['Total_Cholesterol'] * df['iron_lvl']
    df['bmi_iron'] = df['bmi'] * df['iron_lvl']

    # 나눗셈 기반 상호작용
    df['iron_per_stature'] = df['iron_lvl'] / (df['stature_inch'] + 1e-6)
    df['cholesterol_per_age'] = df['Total_Cholesterol'] / (df['age_enzyme_ratio'] + 1e-6)
    df['stature_per_bmi'] = df['stature_inch'] / (df['bmi'] + 1e-6)

    # 덧셈 및 뺄셈 기반 상호작용
    df['iron_plus_cholesterol'] = df['iron_lvl'] + df['Total_Cholesterol']
    df['stature_minus_mass'] = df['stature_inch'] - df['mass_kg']
    df['cholesterol_minus_age'] = df['Total_Cholesterol'] - df['age_enzyme_ratio']

    # 비선형 상호작용
    df['sqrt_iron_cholesterol'] = np.sqrt(df['iron_lvl'] * df['Total_Cholesterol'])
    df['stature_squared'] = df['stature_inch'] ** 2
    df['log_iron_lvl'] = np.log1p(df['iron_lvl'])
    
    return df

# X_train, X_test 데이터에 적용
X_train = create_interaction_features(X_train)
X_test = create_interaction_features(X_test)

drop=[
 'audio_l',
 'audio_r',
#  'bp_tf',
#  'iron_squared',
#  'log_iron_lvl',
#  'sqrt_cholesterol',
#  'sqrt_iron_cholesterol',
#  'stature_squared',
#  'urine_marker_flag',
#  'Metabolic_Syndrome',
#  'Vision_Imbalance',
#  'age',
#  'Total_Cholesterol',
#  'urine_marker',
#  'audio_problem',
#  'mass_kg',
#  'oral_issues'
]
X_train=X_train.drop(drop,axis=1)
X_test=X_test.drop(drop,axis=1)


In [ ]:
X_test

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(random_state=42)
param_grid={
    'n_estimators':[400,600,800,1000],
    'max_depth':[8,10,12,14,16,18,20],
    'class_weight': [None],
    'max_features': ['sqrt','log2'],
    'min_samples_split': [1,2,3,4,5,6]
    
}

randomSearch=RandomizedSearchCV(model, param_grid,scoring='roc_auc',cv=2,verbose=3, n_jobs=4)

randomSearch.fit(X_train,y_train)

best_model = randomSearch.best_estimator_
pred=best_model.predict(X_test)

In [ ]:
test_prob = model.predict_proba(X_test)[:, 1]

# 예측 확률 값 출력
print("Predicted probabilities for X_test:")
print(test_prob)

In [ ]:
# 제출 파일 저장
submission_df = pd.read_csv('sample_submission.csv')
submission_df.health_status = test_prob
submission_df.to_csv('submission.csv', index=False)

In [ ]:
len(train_df.columns)